# Motor & Propeller Sizing Calculations

Calculations for selecting motors and propellers for a quadcopter build.  
All values can be adjusted in the **Design Parameters** cell below — re-run the notebook to update everything.

In [ ]:
import math

# ============================================================
# DESIGN PARAMETERS — edit these and re-run to update results
# ============================================================

mass_kg        = 1.0       # Total drone mass (kg)
num_motors     = 4         # Number of motors
hover_throttle = 0.50      # Target hover throttle fraction (50%)
prop_diameter_in = 7       # Propeller diameter (inches)
C_T            = 0.11      # Thrust coefficient (typical assumption)
rho            = 1.225     # Air density at sea level (kg/m³)
battery_cells  = 4         # LiPo cell count (e.g. 4S)
cell_voltage   = 3.7       # Nominal voltage per cell (V)
load_factor    = 0.7       # RPM load factor (0.6–0.8 typical)

g = 9.81  # m/s²

## 1. Weight & Hover Thrust

$$W = mg$$

$$T_{\text{hover, per motor}} = \frac{W}{n_{\text{motors}}}$$

In [ ]:
W = mass_kg * g
T_hover_per_motor = W / num_motors

print(f"Total weight:              W = {W:.2f} N")
print(f"Hover thrust per motor:    T = {T_hover_per_motor:.2f} N")

## 2. Design Thrust Target

Rule of thumb: hover should occur at ~50% throttle so there's headroom for maneuvering, climbing, and wind.

$$T_{\text{max, per motor}} = \frac{T_{\text{hover, per motor}}}{\text{hover throttle}}$$

In [ ]:
T_max_per_motor = T_hover_per_motor / hover_throttle
T_design = math.ceil(T_max_per_motor)  # round up to nearest whole N

print(f"Required max thrust/motor: {T_max_per_motor:.2f} N")
print(f"Design target per motor:   {T_design} N")

## 3. Propeller Thrust Equation

Standard propeller thrust model:

$$T = C_T \, \rho \, n^2 \, D^4$$

| Symbol | Description |
|--------|-------------|
| $T$ | Thrust (N) |
| $C_T$ | Thrust coefficient |
| $\rho$ | Air density (kg/m³) |
| $n$ | Rotational speed (rev/s) |
| $D$ | Propeller diameter (m) |

## 4. Solving for RPM

Rearranging for $n$:

$$n = \sqrt{\frac{T}{C_T \, \rho \, D^4}}$$

In [ ]:
D_m = prop_diameter_in * 0.0254  # convert inches to metres

n_revs = math.sqrt(T_design / (C_T * rho * D_m**4))
n_rpm  = n_revs * 60

print(f"Prop diameter:   {prop_diameter_in} in = {D_m:.4f} m")
print(f"D⁴:             {D_m**4:.6e}")
print(f"Required speed:  {n_revs:.1f} rev/s")
print(f"                 {n_rpm:.0f} RPM")

## 5. Motor KV Selection

Under load, a brushless motor's RPM drops from its no-load value:

$$\text{RPM}_{\text{loaded}} = \text{load\_factor} \times K_V \times V_{\text{battery}}$$

Solving for $K_V$:

$$K_V = \frac{\text{RPM}_{\text{loaded}}}{\text{load\_factor} \times V_{\text{battery}}}$$

In [ ]:
V_battery = battery_cells * cell_voltage
KV = n_rpm / (load_factor * V_battery)

print(f"Battery voltage ({battery_cells}S): {V_battery:.1f} V")
print(f"Load factor:              {load_factor}")
print(f"Required motor KV:        {KV:.0f} KV")
print(f"\n→ Look for a motor around {round(KV/50)*50}–{round(KV/50)*50 + 100} KV with {prop_diameter_in}\" props on {battery_cells}S.")

## Summary

In [ ]:
print("=" * 50)
print("  MOTOR SIZING SUMMARY")
print("=" * 50)
print(f"  Drone mass:           {mass_kg} kg")
print(f"  Total weight:         {W:.2f} N")
print(f"  Motors:               {num_motors}")
print(f"  Hover thrust/motor:   {T_hover_per_motor:.2f} N")
print(f"  Design thrust/motor:  {T_design} N")
print(f"  Propeller:            {prop_diameter_in} in")
print(f"  Required RPM:         {n_rpm:.0f}")
print(f"  Battery:              {battery_cells}S ({V_battery:.1f} V)")
print(f"  Target motor KV:      ~{KV:.0f} KV")
print("=" * 50)